# AspectBench privacy, extraction, and distribution audit

This notebook is the report layer for the resumable four-GPU audit. It examines whether apparent membership or aspect-name leakage can be separated from ordinary dataset overlap and distribution shift. It never displays or exports article text, UUIDs, aspect names, nearest-neighbour identities, or per-record probabilities.

This is empirical evidence, not a proof of privacy. The released systems are three-class discriminative classifiers: their normal API cannot generate arbitrary training articles, but confidence-based membership inference, entity-property inference, and model inversion remain relevant threat models. Formal differential privacy was not used.

## Experiments and what they establish

1. **Artifact inventory:** verifies release files are tensor-only and contain no accidental corpus, optimizer, cache, or prediction payload.
2. **Tracked release-surface scan:** compares Git-tracked code, notebooks, documentation, and prompt JSON against private corpus fingerprints. It reports file paths and counts—not source text, names, or hashes—and catches examples accidentally packaged outside model weights.
3. **Exact and near-duplicate audit:** hashes normalized UUID/article/article+aspect records and estimates character n-gram nearest-train similarity. This detects data leakage that could otherwise masquerade as model memorization.
4. **Distribution classifier:** predicts train versus validation/test membership from text alone with TF–IDF fitted inside every held-out fold. High AUC means membership attacks are confounded by corpus or split drift.
5. **Black-box membership attacks:** uses gold-label probability, confidence, entropy, margin, MC-dropout disagreement, and a held-out combined attack. Results are run against both validation and test nonmembers and include two-sided permutation tests plus low-FPR operating points.
6. **Aspect distribution:** measures seen/unseen support, aspect-frequency drift, and an aspect-only majority-label baseline. This identifies label predictiveness already present in the dataset.
7. **Aspect-name counterfactuals:** compares original predictions with a synthetic pseudonym and with permuted real aspect names, separately for members and test nonmembers. A member-only increase is more concerning than equal sensitivity in both groups.
8. **Aspect-support property attack:** tests whether confidence reveals that an aspect name occurred in training. This is property inference, not article reconstruction.
9. **Neutral-context probes:** inserts seen and unseen names into a fixed neutral sentence and reports aggregate class skew. These inputs are intentionally out of distribution and are diagnostic only.

A valid retrospective canary-exposure test is impossible because unique canaries were not inserted before training. Stronger shadow-model or LiRA attacks require retained reference checkpoints and are recommended only if the initial audit triggers.

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
import json
import os
from pathlib import Path

import pandas as pd

REPO_ROOT = Path(os.environ.get('ASPECTBENCH_ROOT', '.')).resolve()
if not (REPO_ROOT / 'src' / 'aspectbench').is_dir():
    REPO_ROOT = Path('..').resolve()
RUN_DIR = Path(os.environ.get(
    'PRIVACY_AUDIT_RUN_DIR',
    REPO_ROOT / 'outputs' / 'privacy-audit' / 'privacy-release-audit',
)).resolve()
SUMMARY_PATH = RUN_DIR / 'aggregate-summary.json'
print({'repository': str(REPO_ROOT), 'run_dir': str(RUN_DIR),
       'summary_exists': SUMMARY_PATH.is_file()})

## Running the GPU audit

The notebook intentionally does not launch four GPU workers itself. Use `scripts/6.1-run-privacy-audit-four-gpu.sh`; it maps HBS masked/unmasked and Slovenian masked/unmasked to four GPUs, saves one aggregate report per model, and resumes from `_SUCCESS.json` markers. Then execute this notebook through `scripts/6.3-render-privacy-audit-notebook.sh`. That wrapper preserves both the executed `.ipynb` and an HTML rendering under the ignored run directory.

In [ ]:
if not SUMMARY_PATH.is_file():
    raise FileNotFoundError(
        f'{SUMMARY_PATH} is missing. Run scripts/6.1-run-privacy-audit-four-gpu.sh first.'
    )
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
report_paths = sorted(RUN_DIR.glob('*/*/*/aggregate-report.json'))
reports = [json.loads(path.read_text(encoding='utf-8')) for path in report_paths]
print({'model_reports': len(reports),
       'completed_shards': summary['completed_shards'],
       'failed_markers': len(summary['failed_markers']),
       'review_triggers': len(summary['review_triggers'])})

## 1. Membership inference, with distribution-shift context

In [ ]:
membership = pd.DataFrame(summary['membership_summary'])
display(membership.sort_values(['dataset', 'variant', 'model', 'nonmember_cohort']))

distribution_rows = []
for report in reports:
    for cohort, result in report['data_audit']['text_distribution_classifier'].items():
        distribution_rows.append({
            'dataset': report['dataset'], 'variant': report['variant'],
            'model': report['model'], 'selected_split': report['selected_split'],
            'cohort': cohort,
            'text_only_membership_auc_mean': result['held_out_auc']['mean'],
            'text_only_membership_auc_p90': result['held_out_auc']['p90'],
        })
display(pd.DataFrame(distribution_rows).sort_values(['dataset', 'variant', 'model', 'cohort']))

Interpret membership AUC together with the text-only distribution AUC and the covariate-matched attack column. The matched attack frequency-matches cohorts on sentiment label, article-length quartile, and training-aspect-frequency bucket. A model attack above chance is more concerning when (a) its confidence interval excludes 0.5, (b) it replicates for validation and test nonmembers, (c) it survives covariate matching, and (d) text alone cannot readily distinguish the cohorts. High text-only AUC instead indicates a split-distribution confound that should be resolved before making a memorization claim.

## 2. Release-surface scan, overlap, near duplicates, and aspect distributions

In [ ]:
data_rows, release_surface_rows = [], []
for report in reports:
    audit = report['data_audit']
    exact = audit['exact_overlap']['cross_cohort']['train_vs_test']['normalized_article_and_aspect']
    near_total = audit['near_duplicates']['test']
    near_filtered = audit['near_duplicates_after_exact_exclusion']['test']
    aspects = audit['aspect_distribution']
    surface = audit['tracked_release_surface']
    data_rows.append({
        'dataset': report['dataset'], 'variant': report['variant'],
        'model': report['model'], 'selected_split': report['selected_split'],
        'exact_train_test_rows': exact['right_rows_overlapping_left'],
        'exact_train_test_fraction': exact['right_fraction_overlapping_left'],
        'near_duplicate_fraction_ge_0.95_total': near_total['fraction_at_or_above']['0.95'],
        'near_duplicate_fraction_ge_0.95_after_exact_exclusion': near_filtered['fraction_at_or_above']['0.95'],
        'nearest_similarity_p95_after_exact_exclusion': near_filtered['nearest_train_similarity']['p95'],
        'test_seen_aspect_fraction': aspects['support']['test']['train_seen_fraction'],
        'aspect_distribution_js': aspects['jensen_shannon_divergence_nats']['train_vs_test'],
        'aspect_only_test_macro_f1': aspects['aspect_only_label_baseline']['test']['macro_f1'],
    })
    for match in surface['files_with_matches']:
        release_surface_rows.append({
            'dataset': report['dataset'], 'path': match['path'],
            'exact_article_values': match['exact_article_value_count'],
            'long_shingles': match['distinct_long_shingle_match_count'],
            'aspect_names': match['distinct_aspect_match_count'],
        })
display(pd.DataFrame(data_rows).drop_duplicates().sort_values(['dataset', 'variant', 'model']))
display_or_surface = pd.DataFrame(release_surface_rows).drop_duplicates()
display(display_or_surface.sort_values(['dataset', 'path']) if not display_or_surface.empty else pd.DataFrame({'result': ['No tracked release-surface matches.']}))

## 3. Aspect-name counterfactual and property probes

In [ ]:
counterfactual_rows, property_rows, neutral_rows = [], [], []
for report in reports:
    model_audit = report.get('model_audit', {})
    for probe, result in model_audit.get('aspect_name_counterfactuals', {}).items():
        counterfactual_rows.append({
            'dataset': report['dataset'], 'variant': report['variant'],
            'model': report['model'], 'probe': probe,
            'n': result['n'], 'flip_rate': result['prediction_flip_rate'],
            'mean_js_nats': result['js_divergence_nats']['mean'],
            'p95_js_nats': result['js_divergence_nats']['p95'],
        })
    for cohort, result in model_audit.get('aspect_support_property_inference', {}).items():
        row = {'dataset': report['dataset'], 'variant': report['variant'],
               'model': report['model'], 'cohort': cohort,
               'available': result['available'], 'seen_n': result['seen_n'],
               'unseen_n': result['unseen_n']}
        if result['available']:
            row['strongest_auc'] = max(value['auc'] for value in result['attacks'].values())
        property_rows.append(row)
    for cohort, result in model_audit.get('neutral_context_aspect_probe', {}).items():
        neutral_rows.append({
            'dataset': report['dataset'], 'variant': report['variant'],
            'model': report['model'], 'aspect_group': cohort,
            'n': result['n'], 'mean_confidence': result['mean_confidence'],
            **{f'predicted_fraction_{key}': value for key, value in result['predicted_class_fractions'].items()},
        })
def display_or_message(rows, sort_columns, message):
    frame = pd.DataFrame(rows)
    display(frame.sort_values(sort_columns) if not frame.empty else pd.DataFrame({'result': [message]}))

display_or_message(counterfactual_rows, ['dataset', 'variant', 'model', 'probe'], 'No model counterfactual results in this run.')
display_or_message(property_rows, ['dataset', 'variant', 'model', 'cohort'], 'No property-inference results in this run.')
display_or_message(neutral_rows, ['dataset', 'variant', 'model', 'aspect_group'], 'No neutral-context results in this run.')

For masked models, changing the tagged target should normally have little effect because the target surface form is removed by design. For unmasked models, some sensitivity is expected and is not itself memorization. The more relevant warning is substantially greater sensitivity for training members than test nonmembers, especially alongside a successful membership attack. Neutral-context results are weaker evidence because these synthetic sentences are outside the training distribution.

## 4. Review triggers and an aggregate-only handoff

In [ ]:
trigger_frame = pd.DataFrame(summary['review_triggers'])
display(trigger_frame if not trigger_frame.empty else pd.DataFrame({'result': ['No automatic review triggers.']}))

notebook_summary = {
    'schema_version': 3,
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'source_run': str(RUN_DIR),
    'model_report_count': len(reports),
    'review_trigger_count': len(summary['review_triggers']),
    'failed_marker_count': len(summary['failed_markers']),
    'contains_source_records': False,
    'interpretation': summary['release_interpretation'],
}
summary_output = RUN_DIR / 'notebook-summary.json'
summary_output.write_text(json.dumps(notebook_summary, indent=2) + '\n', encoding='utf-8')
print(notebook_summary)

## Decision rubric

- **Lower concern:** tensor-only artifacts; no restricted-corpus text in tracked files; no train/test exact overlap; low near-duplicate rate; membership AUC intervals include 0.5; combined attacks remain below 0.60; member and test counterfactual sensitivity are similar.
- **Investigate:** AUC ≥ 0.60 with a lower 95% bound above 0.5, replicated across nonmember cohorts; a pronounced member-only counterfactual effect; or confidence that reliably identifies train-seen aspect names.
- **Resolve confounding first:** text-only train/test classifier or near-duplicate detector is strong. Re-match cohorts by source, time, language, length, label, aspect frequency, and similarity before attributing the signal to model parameters.
- **Escalate if needed:** train shadow/reference models on the retained splits and run LiRA-style likelihood ratios. This is much more expensive and still empirical.
- **Never claim:** that a passing audit proves non-memorization, differential privacy, or impossibility of extraction. A defensible statement is that no material leakage was detected by the documented attacks under the tested threat model.